# BETO Fine-tuning — ROCKTEC MIA 2026

Corre `02_scripts/12_beto_finetuning.py` (fine-tuning real de BETO, 5 clases) sobre GPU gratuita de Colab.

**Antes de correr:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → GPU (T4).
2. Asegúrate de haber hecho `git push` de tus cambios locales (incluyendo `02_scripts/12_beto_finetuning.py`) a `origin/main` — este notebook clona el repo desde GitHub, no ve tu filesystem local.

Si todavía no pusheaste, usa la celda opcional al final ("Alternativa: subir archivos manualmente") en vez del `git clone`.

## 1. Verificar GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA disponible:', torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
CUDA disponible: True


## 2. Clonar el repositorio

Requiere que el repo sea público (o que uses un token si es privado — reemplaza la URL por `https://<TOKEN>@github.com/LuisChica18/proyecto-mia-rocktec.git`).

In [3]:
REPO_URL = 'https://github.com/LuisChica18/proyecto-mia-rocktec.git'

import os
if os.path.isdir('proyecto-mia-rocktec'):
    %cd proyecto-mia-rocktec
    !git pull
else:
    !git clone $REPO_URL
    %cd proyecto-mia-rocktec

!git log --oneline -5

Cloning into 'proyecto-mia-rocktec'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 226 (delta 89), reused 161 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (226/226), 6.07 MiB | 4.39 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/proyecto-mia-rocktec
f8715ad (HEAD -> main, origin/main, origin/HEAD) feat: S7 ajuste #1 — BETO embeddings F1=0.6370 vs TF-IDF F1=0.7516, comparacion completa
6db8bb0 feat: S7 ajuste #6 — SHAP y LIME explicabilidad modelo LR 5 clases
9b2bf71 feat: S7 ajuste #2 — holdout set 197 registros (15% estratificado, 5 clases)
dedada6 merge: resolver conflictos manteniendo versiones Fase 1C
00c5ff7 feat: Fase 1C completa — dataset consenso real (Kappa 0.8854), validacion estadistica pareada, scripts 07/08, candidatos QUE/SEG


In [7]:
!pwd

/content/proyecto-mia-rocktec


In [8]:
!ls /content/proyecto-mia-rocktec

01_datos_crudos      04_anotaciones    CHANGELOG.md
02_scripts	     05_documentacion  README.md
03_datos_procesados  06_resultados     requirements.txt


In [ ]:
!ls /content/proyecto-mia-rocktec/02_scripts

## 3. Instalar dependencias

Colab ya trae `torch` con soporte CUDA preinstalado — no lo reinstales desde `requirements.txt` (podría bajarte una build sin CUDA). Instalamos solo lo que falta.

In [5]:
%pip install -q transformers==4.36.0 accelerate==0.25.0 pandas scikit-learn

## 4. Correr el fine-tuning

Usa `04_anotaciones/dataset_consenso_final.csv` (ya está en el repo clonado). Con GPU T4, 5 épocas sobre ~1,050 registros de entrenamiento debería tomar unos pocos minutos.

In [6]:
!python 02_scripts/12_beto_finetuning.py

python3: can't open file '/content/proyecto-mia-rocktec/02_scripts/12_beto_finetuning.py': [Errno 2] No such file or directory


## 5. Ver resultados

In [ ]:
!echo '--- comparacion_tfidf_vs_beto_finetuned.txt ---'
!cat 06_resultados/beto/comparacion_tfidf_vs_beto_finetuned.txt
!echo '--- reporte_beto_finetuned.txt ---'
!cat 06_resultados/beto/reporte_beto_finetuned.txt

## 6. Descargar resultados a tu máquina

Empaqueta los reportes + el checkpoint del modelo y los descarga como zip. El checkpoint (`beto_finetuned_best/`) puede pesar ~440MB — si solo te interesan los reportes de texto, comenta la línea del checkpoint.

In [ ]:
!zip -r resultados_beto_finetuning.zip 06_resultados/beto/reporte_beto_finetuned.txt 06_resultados/beto/comparacion_tfidf_vs_beto_finetuned.txt 06_resultados/modelos/beto_finetuned_best

from google.colab import files
files.download('resultados_beto_finetuning.zip')

Después de descargar, descomprime el zip en la raíz de tu copia local del repo (respetando las rutas `06_resultados/...`) y haz el commit desde VS Code como de costumbre.

---
## Alternativa: subir archivos manualmente (si aún no hiciste `git push`)

Sáltate el paso 2 (clonar) y en su lugar corre esta celda para subir `12_beto_finetuning.py` y `dataset_consenso_final.csv` directo desde tu máquina.

In [ ]:
import os
from google.colab import files

os.makedirs('02_scripts', exist_ok=True)
os.makedirs('04_anotaciones', exist_ok=True)
os.makedirs('06_resultados/beto', exist_ok=True)
os.makedirs('06_resultados/modelos', exist_ok=True)

print('Selecciona 02_scripts/12_beto_finetuning.py')
up = files.upload()
for name in up:
    os.rename(name, f'02_scripts/{name}')

print('Selecciona 04_anotaciones/dataset_consenso_final.csv')
up = files.upload()
for name in up:
    os.rename(name, f'04_anotaciones/{name}')